# 🏀 Predicto Demo — March Machine Learning Mania 2026

**Demonstração completa do pipeline de predição probabilística com calibração de probabilidades.**

---

## 1️⃣ Setup e Importações

In [ ]:
import sys
import os

# Adicionar src ao path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss
from sklearn.calibration import calibration_curve

# Imports do Predicto
from src.config import target_season, backtest_seasons
from src.data import load_games, load_seeds
from src.features import compute_features
from src.ratings import precompute_starting_elo

# Configuração de plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful!")
print(f"Target season: {target_season}")
print(f"Backtest seasons: {backtest_seasons}")

## 2️⃣ Carregamento de Dados

In [ ]:
# Carregar dados de jogos
print("Loading games data...")
games_df = load_games()

print(f"\n📊 Dataset Shape: {games_df.shape}")
print(f"Temporadas: {sorted(games_df['Season'].unique())}")
print(f"\nPrimeiras linhas:")
games_df.head(10)

In [ ]:
# Carregar seeds (seeding do torneio)
print("Loading tournament seeds...")
seeds_df = load_seeds()

print(f"\n🏆 Seeds Shape: {seeds_df.shape}")
print(f"\nAmostra de seeds (Season 2024):")
seeds_df[seeds_df['Season'] == 2024].head(10)

## 3️⃣ ELO Ratings — Sistema de Classificação Cross-Temporal

In [ ]:
# Computar ELO ratings para todos os times
print("Computing ELO ratings with cross-seasonal carry-forward...")
elo_history = precompute_starting_elo()

print(f"\n⭐ ELO History Shape: {elo_history.shape}")
print(f"\nELO ratings sample (Team=1143, 2023-2025):")
elo_sample = elo_history[(elo_history['TeamID'] == 1143) & (elo_history['Season'] >= 2023)]
print(elo_sample[['Season', 'TeamID', 'ELO']].sort_values('Season'))

In [ ]:
# Visualizar distribuição de ELO ratings por temporada
elo_stats = elo_history.groupby('Season')['ELO'].describe()
print("\n📊 ELO Rating Statistics by Season:")
print(elo_stats)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
elo_history.boxplot(column='ELO', by='Season', ax=ax)
plt.title('ELO Ratings Distribution by Season')
plt.suptitle('')
plt.xlabel('Season')
plt.ylabel('ELO Rating')
plt.tight_layout()
plt.show()

## 4️⃣ Feature Engineering

In [ ]:
# Computar features para um season específico
test_season = 2024
print(f"Computing features for season {test_season}...")

# Filtrar games da temporada
season_games = games_df[games_df['Season'] == test_season].copy()
print(f"\nGames in season {test_season}: {len(season_games)}")

# Computar features (simplificado para demo)
print(f"\nSample games com features:")
season_games[['Season', 'WTeamID', 'LTeamID', 'WScore', 'LScore']].head()

## 5️⃣ Métricas de Desempenho

In [ ]:
# Exemplo de métricas de calibração
# Criar predições dummy para demonstração
np.random.seed(42)

# Simular probabilidades preditas e labels reais
n_samples = 1000
y_true = np.random.binomial(1, 0.5, n_samples)
y_pred_uncalibrated = np.random.uniform(0, 1, n_samples)

# Adicionar correlação com y_true
y_pred_uncalibrated = (y_pred_uncalibrated + 2 * y_true) / 3
y_pred_uncalibrated = np.clip(y_pred_uncalibrated, 0, 1)

# Aplicar calibração simples (Platt scaling)
from sklearn.linear_model import LogisticRegression
calib_model = LogisticRegression()
calib_model.fit(y_pred_uncalibrated.reshape(-1, 1), y_true)
y_pred_calibrated = calib_model.predict_proba(y_pred_uncalibrated.reshape(-1, 1))[:, 1]

# Calcular métricas
brier_uncal = brier_score_loss(y_true, y_pred_uncalibrated)
brier_cal = brier_score_loss(y_true, y_pred_calibrated)
auc = roc_auc_score(y_true, y_pred_calibrated)
logloss = log_loss(y_true, y_pred_calibrated)

print("📊 PERFORMANCE METRICS")
print("=" * 50)
print(f"Brier Score (Uncalibrated): {brier_uncal:.4f}")
print(f"Brier Score (Calibrated):   {brier_cal:.4f}")
print(f"Improvement:                {brier_uncal - brier_cal:.4f}")
print(f"\nAUC-ROC:                    {auc:.4f}")
print(f"Log Loss:                   {logloss:.4f}")
print("=" * 50)

## 6️⃣ Calibration Curve

In [ ]:
# Plotar calibration curve antes e depois
prob_true_uncal, prob_pred_uncal = calibration_curve(y_true, y_pred_uncalibrated, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_true, y_pred_calibrated, n_bins=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Uncalibrated
ax1.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
ax1.plot(prob_pred_uncal, prob_true_uncal, 'o-', label='Uncalibrated', linewidth=2, markersize=8)
ax1.set_xlabel('Mean Predicted Probability')
ax1.set_ylabel('Fraction of Positives')
ax1.set_title(f'Uncalibrated (Brier: {brier_uncal:.4f})')
ax1.legend()
ax1.grid(alpha=0.3)

# Calibrated
ax2.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
ax2.plot(prob_pred_cal, prob_true_cal, 'o-', label='Calibrated (Platt)', linewidth=2, markersize=8, color='green')
ax2.set_xlabel('Mean Predicted Probability')
ax2.set_ylabel('Fraction of Positives')
ax2.set_title(f'Calibrated (Brier: {brier_cal:.4f})')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Calibration improved significantly!")

## 7️⃣ Configuração do Pipeline

In [ ]:
# Ver configuração do projeto
print("📋 PREDICTO CONFIGURATION")
print("=" * 50)
print(f"Target Season (para submission):    {target_season}")
print(f"Backtest Seasons:                   {backtest_seasons}")
print(f"Número de folds:                    {len(backtest_seasons)}")
print(f"\nELO Carry-Forward Factor:          0.75 (padrão)")
print(f"ELO Initial Rating:                 1500")
print(f"ELO K-Factor Base:                  32")
print(f"\nXGBoost n_estimators:              500")
print(f"XGBoost max_depth:                  6")
print(f"XGBoost learning_rate:              0.1")
print("=" * 50)

## 8️⃣ Próximas Etapas

In [ ]:
print("\n🚀 PARA EXECUTAR O PIPELINE COMPLETO:")
print("\nOpção 1 - Full pipeline (backtest + submit):")
print("  $ python scripts/run_pipeline_2026.py")
print("\nOpção 2 - Apenas backtest:")
print("  $ python scripts/run_pipeline_2026.py --mode backtest")
print("\nOpção 3 - Apenas submission:")
print("  $ python scripts/run_pipeline_2026.py --mode submit")
print("\n" + "="*50)
print("📚 Documentação completa em PROVENANCE.md")
print("🐛 Bugs e fixes em AUDITORIA_TECNICA_v3.md")
print("📝 Histórico de mudanças em CHANGELOG.md")
print("="*50)

---

## 📌 Resumo da Demo

✅ **Cobertura:**
1. Setup e importações
2. Carregamento de dados (games + seeds)
3. Sistema de ELO ratings com carry-forward
4. Feature engineering
5. Métricas de desempenho (Brier Score, AUC, Log Loss)
6. Calibration curves antes/depois
7. Configuração do pipeline
8. Instruções de próximas etapas

**Version:** 3.0  
**Last Updated:** March 2026  
**Author:** João Pedro Passos Tocantins  
**License:** MIT